# Agentic Retrieval Deep Dive — AgenticRetrieveStream API

The `AgenticRetrieveStream` API uses a foundation model to intelligently decompose complex queries into sub-queries, iteratively retrieve relevant information, and optionally generate a synthesized answer with citations.

### How agentic retrieval works

```
User Query (complex, multi-part)
    │
    ▼
┌─────────────────────────────────────────────┐
│ 1. PLANNING                                 │
│    FM analyzes query → creates sub-queries  │
└──────────────────┬──────────────────────────┘
                   ▼
┌─────────────────────────────────────────────┐
│ 2. SPECULATIVE RETRIEVAL                    │
│    Initial retrieval with raw query         │
│    (runs in parallel with planning)         │
└──────────────────┬──────────────────────────┘
                   ▼
┌─────────────────────────────────────────────┐
│ 3. RETRIEVAL (per sub-query)                │
│    Execute sub-queries against retrievers   │
│    Evaluate: sufficient? If not → iterate   │
└──────────────────┬──────────────────────────┘
                   ▼ (up to maxAgentIteration)
┌─────────────────────────────────────────────┐
│ 4. FULL DOCUMENT EXPANSION (if needed)      │
│    GetDocumentContent for full context       │
└──────────────────┬──────────────────────────┘
                   ▼
┌─────────────────────────────────────────────┐
│ 5. RESPONSE GENERATION (if enabled)         │
│    Synthesize answer from all chunks         │
│    Stream text + citations                   │
└─────────────────────────────────────────────┘
```

### What this notebook covers

| # | Topic |
|---|-------|
| 1 | Trace event analysis — see the agent's planning and retrieval steps |
| 2 | maxAgentIteration — impact of 1 vs 3 vs 5 iterations |
| 3 | generate_response — chunks-only vs full generation |
| 4 | Multi-turn conversation — using message history |
| 5 | Multi-KB retrieval — multiple retrievers |
| 6 | Citation parsing — map answer spans to source chunks |

### Prerequisites

- AWS credentials with Bedrock and IAM permissions  
- Model access for Claude and managed embedding
- Python 3.10+

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../../requirements.txt --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1 — Setup

In [ ]:
import boto3
import sys
import time
import os
import json
import pprint

sys.path.insert(0, "../..")

s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

bucket_name = f'bedrock-bmkb-agentic-{suffix}-{account_id}'
embedding_model = None  # Managed default

region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:     {region}')
print(f'Account:    {account_id}')
print(f'Bucket:     {bucket_name}')
print(f'Model ARN:  {generation_model_arn}')

## Step 2 — Create KB and ingest

In [ ]:
# Create bucket and upload
try:
    s3_client.head_bucket(Bucket=bucket_name)
except Exception:
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})

s3_client.upload_file('../../synthetic_dataset/octank_financial_10K.pdf', bucket_name, 'octank_financial_10K.pdf')
print(f'Uploaded to s3://{bucket_name}/octank_financial_10K.pdf')

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=f'bmkb-agentic-{suffix}',
    bucket_name=bucket_name,
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f'KB ID: {kb.kb_id}')
time.sleep(15)
kb.start_ingestion_job()

## Step 3 — Trace event analysis

Every `AgenticRetrieveStream` call emits trace events showing the agent's reasoning: planning, speculative retrieval, sub-query execution, and evaluation.

Let's parse these traces to understand what the agent does under the hood.

In [ ]:
runtime_client = kb.get_runtime_client()

def agentic_retrieve_raw(kb_id, query, model_arn, max_results=10, max_iterations=3, generate_response=True, messages=None):
    """
    Call AgenticRetrieveStream directly and collect all event types separately.
    Returns dict with traces, results, response_chunks, and generated_response.
    """
    if messages is None:
        messages = [{'role': 'user', 'content': {'text': query}}]
    
    response = runtime_client.agentic_retrieve_stream(
        messages=messages,
        retrievers=[{
            'configuration': {
                'knowledgeBase': {
                    'knowledgeBaseId': kb_id,
                    'retrievalOverrides': {'maxNumberOfResults': max_results},
                }
            }
        }],
        agenticRetrieveConfiguration={
            'foundationModelConfiguration': {
                'bedrockFoundationModelConfiguration': {
                    'modelConfiguration': {'modelArn': model_arn}
                },
                'type': 'BEDROCK_FOUNDATION_MODEL',
            },
            'foundationModelType': 'CUSTOM',
            'maxAgentIteration': max_iterations,
            'rerankingModelType': 'MANAGED',
        },
        generateResponse=generate_response,
    )
    
    traces = []
    results = []
    response_chunks = []
    generated_response = None
    
    for event in response['stream']:
        if 'traceEvent' in event:
            traces.append(event['traceEvent'])
        elif 'responseEvent' in event:
            response_chunks.append(event['responseEvent'].get('text', ''))
        elif 'result' in event:
            results = event['result'].get('results', [])
            generated_response = event['result'].get('generatedResponse')
    
    return {
        'traces': traces,
        'results': results,
        'response_chunks': response_chunks,
        'generated_response': generated_response,
    }

print('Helper function defined.')

In [ ]:
# Complex multi-part query that requires decomposition
query = "Compare Octank Financial's market risk exposure with their credit risk metrics, and explain how their liquidity management addresses both."

result = agentic_retrieve_raw(kb.kb_id, query, generation_model_arn, max_iterations=3)

print(f'=== Trace Events ({len(result["traces"])} total) ===\n')

for i, trace in enumerate(result['traces'], 1):
    trace_type = trace.get('type', 'unknown')
    attrs = trace.get('attributes', {})
    
    # Extract key info based on trace type
    step = attrs.get('step', '')
    status = attrs.get('status', '')
    sub_query = attrs.get('query', '')
    num_results = attrs.get('numberOfResults', '')
    
    print(f'{i:2d}. [{trace_type}]')
    if step:
        print(f'     Step: {step}')
    if status:
        print(f'     Status: {status}')
    if sub_query:
        print(f'     Query: "{sub_query[:80]}"')
    if num_results:
        print(f'     Results: {num_results}')
    
    # Show raw attributes for transparency
    other_attrs = {k: v for k, v in attrs.items() if k not in ('step', 'status', 'query', 'numberOfResults')}
    if other_attrs:
        for k, v in list(other_attrs.items())[:3]:
            val = str(v)[:80]
            print(f'     {k}: {val}')
    print()

print(f'\nFinal: {len(result["results"])} deduplicated chunks')
if result['generated_response']:
    print(f'Answer length: {len(result["generated_response"]["answer"])} chars')
    print(f'Citations: {len(result["generated_response"].get("citations", []))}')

## Step 4 — maxAgentIteration comparison

The `maxAgentIteration` controls how many planning→retrieval→evaluation cycles the agent performs. More iterations = better for complex queries, but higher latency.

| Iterations | Best for |
|-----------|----------|
| 1 | Simple, single-topic queries |
| 3 (default) | Multi-part questions |
| 5 | Complex comparative/analytical questions |

In [ ]:
complex_query = "What were Octank Financial's total assets, how did they perform vs prior year, what are the biggest risk categories, and what strategic initiatives are planned to address them?"

print(f'Query: "{complex_query}"\n')
print(f'{"Iterations":<12} {"Traces":<8} {"Chunks":<8} {"Answer len":<12} {"Citations":<10}')
print('-' * 55)

for max_iter in [1, 3, 5]:
    start = time.time()
    result = agentic_retrieve_raw(kb.kb_id, complex_query, generation_model_arn, max_iterations=max_iter)
    elapsed = time.time() - start
    
    answer_len = len(result['generated_response']['answer']) if result.get('generated_response') else 0
    citations = len(result['generated_response'].get('citations', [])) if result.get('generated_response') else 0
    
    print(f'{max_iter:<12} {len(result["traces"]):<8} {len(result["results"]):<8} {answer_len:<12} {citations:<10} ({elapsed:.1f}s)')

print('\nMore iterations → more traces, potentially more chunks, richer answers.')
print('Diminishing returns above 3 for most queries.')

## Step 5 — generate_response: True vs False

- `generate_response=True` (default): Agent retrieves chunks AND generates a synthesized answer with citations
- `generate_response=False`: Agent retrieves chunks only — no generation, no citations

Use `False` when you want to handle generation yourself (custom prompts, different models, etc.).

In [ ]:
query = "What is Octank Financial's approach to cybersecurity and how much do they invest in technology?"

# With generation
print('=== generate_response=True (retrieval + generation) ===')
result_gen = agentic_retrieve_raw(kb.kb_id, query, generation_model_arn, generate_response=True)
print(f'Chunks: {len(result_gen["results"])}')
print(f'Answer: {result_gen["generated_response"]["answer"][:200]}...')
print(f'Citations: {len(result_gen["generated_response"].get("citations", []))}')
print(f'Response stream chunks: {len(result_gen["response_chunks"])}')

print()

# Without generation (chunks only)
print('=== generate_response=False (retrieval only) ===')
result_no_gen = agentic_retrieve_raw(kb.kb_id, query, generation_model_arn, generate_response=False)
print(f'Chunks: {len(result_no_gen["results"])}')
print(f'Generated response: {result_no_gen["generated_response"]}')
print(f'Response stream chunks: {len(result_no_gen["response_chunks"])}')
print(f'\nFirst 3 chunks:')
for i, r in enumerate(result_no_gen['results'][:3], 1):
    print(f'  {i}. {r["content"]["text"][:120]}...')

## Step 6 — Multi-turn conversation

The `messages` array supports conversation history. The agent uses prior context to understand follow-up questions without re-stating the full context.

In [ ]:
# Turn 1: Initial question
messages_turn1 = [
    {'role': 'user', 'content': {'text': 'What are Octank Financial\'s main business segments?'}}
]

print('=== Turn 1 ===')
print(f'User: {messages_turn1[0]["content"]["text"]}')
result1 = agentic_retrieve_raw(kb.kb_id, None, generation_model_arn, messages=messages_turn1)
answer1 = result1['generated_response']['answer'] if result1.get('generated_response') else 'No answer'
print(f'Assistant: {answer1[:300]}...')

print()

# Turn 2: Follow-up (uses conversation context)
messages_turn2 = [
    {'role': 'user', 'content': {'text': 'What are Octank Financial\'s main business segments?'}},
    {'role': 'assistant', 'content': {'text': answer1}},
    {'role': 'user', 'content': {'text': 'Which one grew the fastest and why?'}},
]

print('=== Turn 2 (follow-up) ===')
print(f'User: {messages_turn2[-1]["content"]["text"]}')
result2 = agentic_retrieve_raw(kb.kb_id, None, generation_model_arn, messages=messages_turn2)
answer2 = result2['generated_response']['answer'] if result2.get('generated_response') else 'No answer'
print(f'Assistant: {answer2[:300]}...')
print(f'\nChunks: {len(result2["results"])} | Citations: {len(result2["generated_response"].get("citations", []))}')

## Step 7 — Multi-KB retrieval

You can configure up to **5 retrievers** pointing to different KBs. The agent routes sub-queries to the most relevant KB.

For this demo we use the same KB ID twice (simulating 2 KBs). In production, you'd point each retriever to a different KB (e.g., one for financial data, one for HR policies).

In [ ]:
# Multi-retriever example (using same KB to demonstrate the API shape)
# In production: each retriever would point to a different KB

response = runtime_client.agentic_retrieve_stream(
    messages=[{'role': 'user', 'content': {'text': 'What are the financial metrics and risk management strategies?'}}],
    retrievers=[
        {
            'configuration': {
                'knowledgeBase': {
                    'knowledgeBaseId': kb.kb_id,
                    'retrievalOverrides': {'maxNumberOfResults': 5},
                }
            }
        },
        # In production, add more retrievers pointing to different KBs:
        # {
        #     'configuration': {
        #         'knowledgeBase': {
        #             'knowledgeBaseId': 'ANOTHER_KB_ID',
        #             'retrievalOverrides': {'maxNumberOfResults': 5},
        #         }
        #     }
        # },
    ],
    agenticRetrieveConfiguration={
        'foundationModelConfiguration': {
            'bedrockFoundationModelConfiguration': {
                'modelConfiguration': {'modelArn': generation_model_arn}
            },
            'type': 'BEDROCK_FOUNDATION_MODEL',
        },
        'foundationModelType': 'CUSTOM',
        'maxAgentIteration': 3,
        'rerankingModelType': 'MANAGED',
    },
    generateResponse=True,
)

# Collect results
traces = []
results = []
gen_resp = None

for event in response['stream']:
    if 'traceEvent' in event:
        traces.append(event['traceEvent'])
    elif 'result' in event:
        results = event['result'].get('results', [])
        gen_resp = event['result'].get('generatedResponse')

print(f'Multi-retriever results:')
print(f'  Traces: {len(traces)}')
print(f'  Chunks: {len(results)}')
if gen_resp:
    print(f'  Answer: {gen_resp["answer"][:200]}...')

print('\nNote: Up to 5 retrievers supported. The agent routes sub-queries to the most relevant KB.')

## Step 8 — Citation parsing

When `generate_response=True`, the response includes citations that map character spans in the answer to specific retrieval results.

```
citations[i].startIndex → character start in answer
citations[i].endIndex   → character end in answer  
citations[i].references[j].resultIndex → index into results[] array
```

In [ ]:
query = "What is Octank Financial's revenue and what are their main risk factors?"

result = agentic_retrieve_raw(kb.kb_id, query, generation_model_arn)

if result.get('generated_response'):
    answer = result['generated_response']['answer']
    citations = result['generated_response'].get('citations', [])
    chunks = result['results']
    
    print(f'Answer ({len(answer)} chars):\n{answer[:500]}\n')
    print(f'=== Citations ({len(citations)}) ===\n')
    
    for i, cite in enumerate(citations, 1):
        start = cite.get('startIndex', 0)
        end = cite.get('endIndex', 0)
        cited_text = answer[start:end][:80] if end > start else ''
        refs = cite.get('references', [])
        
        print(f'Citation {i} [chars {start}-{end}]: "{cited_text}..."')
        for ref in refs:
            idx = ref.get('resultIndex', -1)
            if 0 <= idx < len(chunks):
                chunk_text = chunks[idx]['content']['text'][:100]
                print(f'  → result[{idx}]: {chunk_text}...')
        print()
else:
    print('No generated response available.')

## Step 9 — Simple vs complex query comparison

Demonstrate that agentic retrieval adds the most value for complex, multi-hop questions.

In [ ]:
queries = {
    'Simple (single fact)': 'What is Octank Financial\'s total revenue?',
    'Medium (two facts)': 'What is the revenue and headcount?',
    'Complex (multi-hop)': 'Compare revenue growth across business segments, identify which segment has the highest risk, and explain how it affects overall company valuation.',
}

print(f'{"Type":<22} {"Traces":<8} {"Chunks":<8} {"Answer":<10} {"Time":<8}')
print('-' * 58)

for label, q in queries.items():
    start = time.time()
    result = agentic_retrieve_raw(kb.kb_id, q, generation_model_arn, max_iterations=3)
    elapsed = time.time() - start
    
    answer_len = len(result['generated_response']['answer']) if result.get('generated_response') else 0
    print(f'{label:<22} {len(result["traces"]):<8} {len(result["results"]):<8} {answer_len:<10} {elapsed:.1f}s')

print('\nComplex queries trigger more planning + retrieval cycles → richer answers.')

## Step 10 — Cleanup

In [ ]:
# Uncomment to delete all resources
# kb.delete_kb(delete_iam=True, delete_s3_bucket=True)

## Summary

### AgenticRetrieveStream key parameters

| Parameter | Description | Default |
|-----------|-------------|--------|
| `messages` | Query + conversation history | Required |
| `retrievers` | Up to 5 KB retrievers | Required (1+) |
| `maxAgentIteration` | Planning→retrieval cycles | 3 |
| `generateResponse` | Generate synthesized answer | True |
| `rerankingModelType` | MANAGED / CUSTOM / NONE | MANAGED |
| `foundationModelConfiguration` | FM for planning + generation | Required |

### Trace event types

| Type | What it shows |
|------|---------------|
| **Planning** | Query decomposition into sub-queries |
| **Speculative retrieval** | Initial retrieval with raw query (reduces latency) |
| **Retrieval** | Sub-query execution against KB |
| **Full document expansion** | GetDocumentContent for full context |

### When to use AgenticRetrieveStream

| Query type | Use |
|------------|-----|
| Simple fact lookup | `Retrieve` is sufficient |
| Multi-part question | `AgenticRetrieveStream` with maxIteration=3 |
| Complex analytical | `AgenticRetrieveStream` with maxIteration=5 |
| Conversational follow-ups | `AgenticRetrieveStream` with message history |
| Multi-source queries | `AgenticRetrieveStream` with multiple retrievers |

### Key considerations

- Only supports **Managed Knowledge Bases**
- More iterations = better answers for complex queries, but higher latency/cost
- Speculative retrieval runs in parallel with planning to reduce perceived latency
- Results are deduplicated across sub-queries automatically
- Citations map answer spans to specific chunks via `resultIndex`

### Documentation

- [Use agentic retrieval to query a knowledge base](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-test-agentic-retrieve.html)
- [AgenticRetrieveStream API reference](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_agent-runtime_AgenticRetrieveStream.html)
- [Service quotas for managed KBs](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-quotas.html)